# Preprocessing

## Importing modules

In [ ]:
import pandas as pd
import numpy as np

## 1. Importing and merging data

- Importing the clinical dataset as X_clinical, label data as Y.
- We are merging them as clinical_dataset: left join on 'ID'. The EDA reveals that IDs in both datasets are unique and all patients in the X_clinical dataset have a record in the labels dataset and vice versa.
- We are setting the index to 'ID'.

In [ ]:
X_clinical = pd.read_csv('../data/raw/X_train/clinical_train.csv')
Y = pd.read_csv('../data/raw/target_train.csv')

clinical_dataset = pd.merge(X_clinical, Y, how='left', on='ID', validate='one_to_one')
clinical_dataset.set_index('ID', inplace=True)
print(f"Size of the merged dataset: {clinical_dataset.shape}")
clinical_dataset.head()

## 2. Missing data

### 2.1 Missing data in the labels

In [ ]:
Y_missing = pd.DataFrame({'Number of missing data in the labels' : clinical_dataset[['OS_YEARS', 'OS_STATUS']].isna().sum(), 'Proportion of missing data in the labels (%)' : (clinical_dataset[['OS_YEARS', 'OS_STATUS']].isna().sum()/clinical_dataset[['OS_YEARS', 'OS_STATUS']].shape[0])*100})
Y_missing

As seen in the EDA 150 values are missing in the labels' dataset, the analysis of these values shows that  patients missing OS_YEARS are exactly those missing OS_STATUS (see EDA §2.1) . These patients do not brig any information to our study, we are dropping them. \
The 53 censored patients with a follow-up duration of 0 (OS_YEARS, OS_STATUS) are kept. The IPCW C-index handles them.

In [ ]:
n_dropped = clinical_dataset['OS_YEARS'].isna().sum()
print(f"{n_dropped} patients excluded")
clinical_dataset.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
print(f"New size of the clinical dataset: {clinical_dataset.shape}")
clinical_dataset[['OS_YEARS', 'OS_STATUS']].isna().sum()

In [ ]:
assert clinical_dataset.shape == (3173, 10)

### 2.2 Missing data in the features

A quick reminder of what we found during the EDA :

There is a lot of missing values for the MONOCYTES (18.1%) CYTOGENETICS (11.6%) and WBC (8.2%) features.
- The REL and UOXF centers did not collect any MONOCYTES data. We won't be able to replace these missing values with values estimated per center.
- Missing data are over-represented in some centers: HMS & DUTH.
- Missing data in the MONOCYTES, WBC, ANC and PLT features seem to vary by CENTER
- Missing data in the BM_BLAST and HB features seem to be more randomly distributed \
\
**In the clinical dataset, missing values are not randomly distributed. We must take this into account when replacing missing data.** \
\
As missing features may be links to CENTER, we are keeping this feature, and we will encode it in OneHot. (No CENTER values are missing in the dataset)

We will estimate the missing values with the median in the pipeline to avoid any leaks. Missingness being potentially informative (center-structures), we add missing indicator.

In [ ]:
hemo_param = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
for var in hemo_param:
    clinical_dataset[var+'_MISSING'] = clinical_dataset[var].isna()

clinical_dataset.head()

In [ ]:
clinical_dataset[['BM_BLAST_MISSING', 'WBC_MISSING', 'ANC_MISSING', 'MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING']].mean()

## 3. CYTOGENETICS

We are creating a feature to indicate whether the karyotype is normal or abnormal. \
The feature takes a dataframe with the karyotype and returns the dataframe with four new features:
- CYTO_NORMAL : True if the karyotype is normal, False otherwise.
- CYTO_ABNORMAL: True if the karyotype is not normal (including if it is complex) and not an na, False otherwise.
- CYTO_COMPLEX: True if the karyotype specifies complex, False otherwise. Complex is a subset of abnormal. complex karyotypes are also abnormal (both flags are true).
- CYTO_MISSING: True if the karyotype is missing (na), False otherwise.

In [ ]:
def add_cytogenetics_flags(data_frame):
    cyto = data_frame['CYTOGENETICS']
    # Creating masks
    is_missing = cyto.isna()
    is_normal = cyto.str.match(r'46,(xx|xy)(\[\d+\])?$', case=False, na=False)
    is_complex = cyto.str.match('complex', case=False, na=False)

    # Creating the normal feature
    data_frame['CYTO_NORMAL'] = is_normal & ~is_missing
    # Creating the complex feature
    data_frame['CYTO_COMPLEX'] = is_complex & ~is_missing
    # Creating the abnormal feature
    data_frame['CYTO_ABNORMAL'] = ~is_normal & ~is_missing
    # Creating the missing feature
    data_frame['CYTO_MISSING'] = is_missing

    return data_frame

We are dropping the CYTOGENETICS columns as the information is now encoded in our four new features.

In [ ]:
clinical_dataset = add_cytogenetics_flags(clinical_dataset)
clinical_dataset.drop(columns= 'CYTOGENETICS', inplace=True)
clinical_dataset.head()

In [ ]:
clinical_dataset.value_counts()

## 4. Log transformation

Transforming BM_BLAST, WBC, ANC, MONOCYTES and PLT with log1p as these features are right-skewed.

In [ ]:
hemo_param_log = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'PLT']
for var in hemo_param_log:
    clinical_dataset[var] = np.log1p(clinical_dataset[var])

clinical_dataset.rename(columns={v: f'{v}_LOG' for v in hemo_param_log}, inplace=True)
clinical_dataset.head()

## 5. Export and transforming boolean features

First we are transforming our boolean features in binary features to avoid any problems with the extraction.

In [ ]:
bool_cols = ['CYTO_NORMAL', 'CYTO_COMPLEX', 'CYTO_ABNORMAL', 'CYTO_MISSING', 'BM_BLAST_MISSING', 'WBC_MISSING', 'ANC_MISSING', 'MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING']

for col in bool_cols:
    clinical_dataset[col] = clinical_dataset[col].astype(int)

clinical_dataset.head()

Verifying the right structure of the processed dataframe.

In [ ]:
expected_cols = {'CENTER', 'BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB', 'PLT_LOG',
                 'OS_YEARS', 'OS_STATUS',
                 'CYTO_NORMAL', 'CYTO_COMPLEX', 'CYTO_ABNORMAL', 'CYTO_MISSING',
                 'BM_BLAST_MISSING', 'WBC_MISSING', 'ANC_MISSING', 'MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING'}
assert set(clinical_dataset.columns) == expected_cols
assert clinical_dataset.shape == (3173, 19)

Exporting the dataset into a csv.

In [ ]:
clinical_dataset.to_csv('../data/processed/clinical_train_processed.csv', index=True)

Displaying the dataset and verifying the index.

In [ ]:
df_read = pd.read_csv('../data/processed/clinical_train_processed_v1.csv', index_col=0)
print(f"The index is the patients IDs: {df_read.index.name == 'ID'}")
print(f"The shape of the processed dataset is: {df_read.shape}")
df_read.columns